# Simulated benchmark

Run from `reproduce/Reproduce Simulated Results/` using the project environment.

SVG plots save under `roc_plots/experiment1/`, `experiment2/`, and `experiment3/`.
The combined method/task results and simulation settings save to `roc_plots/roc_metrics.tsv`.


In [ ]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

import decoupler as dc
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
from matplotlib.ticker import MultipleLocator
from scipy.sparse import csr_matrix
from sklearn.metrics import auc, roc_curve

from src.z_aggregate import run_z_aggregate


## Plot settings


In [ ]:
FIGURE_SIZE = (4, 4)
PLOT_STYLE = {
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.labelsize": 10,
    "axes.linewidth": 0.8,
    "axes.edgecolor": "0.25",
    "axes.facecolor": "white",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "0.8",
    "grid.alpha": 0.3,
    "grid.linewidth": 0.5,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "xtick.direction": "out",
    "xtick.bottom": True,
    "ytick.direction": "out",
    "ytick.left": True,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "svg.fonttype": "path",
    "pdf.fonttype": 42,
}

MIN_TARGETS = 5
METHODS = {
    "z-aggregate": {"color": "tab:green"},
    "viper": {"color": "tab:orange"},
    "ulm": {"color": "tab:blue"},
    "mlm": {"color": "tab:purple"},
    "zscore": {"color": "tab:red"},
}


## ROC plotting


In [ ]:
def plot_simulated_roc(adata, ground_truth, filename_prefix=None):
    tasks = [(1, "Activated TFs", "activated"), (-1, "Inhibited TFs", "inhibited")]

    results = []
    with plt.rc_context(PLOT_STYLE):
        for direction, title, suffix in tasks:
            output_file = f"{filename_prefix}_{suffix}.svg" if filename_prefix else None
            curves = []
            for method in METHODS:
                score_key = f"score_{method}"
                if score_key not in adata.obsm:
                    continue
                scores = adata.obsm[score_key]
                cells = ground_truth.index.intersection(scores.index)
                tfs = ground_truth.columns.intersection(scores.columns)
                if cells.empty or tfs.empty:
                    continue

                y_true = (
                    ground_truth.loc[cells, tfs].to_numpy().ravel() == direction
                ).astype(int)
                if not y_true.any():
                    continue
                y_score = direction * scores.loc[cells, tfs].to_numpy().ravel()
                fpr, tpr, _ = roc_curve(y_true, y_score)
                roc_auc = auc(fpr, tpr)
                curves.append((method, fpr, tpr, roc_auc))
                results.append(
                    {
                        "Task": suffix,
                        "Method": method,
                        "ROC_AUC": roc_auc,
                        "N_Cells": len(cells),
                        "N_TFs": len(tfs),
                        "N_Pos": int(y_true.sum()),
                        "N_Neg": int(len(y_true) - y_true.sum()),
                    }
                )

            if not curves:
                print(f"Skipping {title}: no valid ground truth data found.")
                continue

            fig, ax = plt.subplots(figsize=FIGURE_SIZE)
            fig.subplots_adjust(left=0.16, right=0.916, bottom=0.16, top=0.84)
            ax.set_box_aspect(1)
            ax.set_anchor("W")
            ax.plot(
                [0, 1],
                [0, 1],
                color="0.6",
                linestyle=(0, (4, 3)),
                linewidth=1,
                label="_nolegend_",
                zorder=0,
            )
            for method, fpr, tpr, roc_auc in sorted(
                curves, key=lambda curve: curve[3], reverse=True
            ):
                ax.plot(
                    fpr,
                    tpr,
                    label=f"{method} ({roc_auc:.4f})",
                    color=METHODS[method]["color"],
                    linewidth=1.2,
                    alpha=0.6,
                    zorder=2,
                )

            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.xaxis.set_major_locator(MultipleLocator(0.2))
            ax.yaxis.set_major_locator(MultipleLocator(0.2))
            ax.tick_params(length=3, width=0.8, pad=3)
            ax.text(
                0.5,
                1.03,
                title,
                transform=ax.transAxes,
                fontsize=8,
                color="0.35",
                ha="center",
                va="bottom",
                zorder=5,
            )
            ax.set_xlabel("False positive rate", fontsize=7)
            ax.set_ylabel("True positive rate", fontsize=7)
            ax.legend(
                loc="lower right",
                frameon=True,
                facecolor="white",
                edgecolor="0.85",
                framealpha=0.95,
                fontsize=8,
                handlelength=2.4,
                handletextpad=0.6,
                labelspacing=0.7,
                borderpad=0.5,
            )

            if output_file:
                fig.savefig(output_file, dpi=300, bbox_inches="tight", pad_inches=0.06)
                print(f"Saved to {output_file}")
            plt.close(fig)

    return pd.DataFrame(results)


## Simulation and scoring


In [ ]:
def plot_simulated_benchmark(params, is_noisy_prior=False, save_plot=False):
    with tempfile.TemporaryDirectory() as temp_dir:
        params_file = Path(temp_dir) / "params.json"
        params_file.write_text(json.dumps(params))
        subprocess.run(
            [
                sys.executable,
                "simulated_data_generator.py",
                "--params-file",
                str(params_file),
            ],
            check=True,
        )
    data_dir = Path(params["output_dir"])
    expression = pd.read_csv(data_dir / params["output_exp_file"], sep="\t", index_col=0)
    adata = sc.AnnData(
        X=csr_matrix(expression.to_numpy()),
        obs=pd.DataFrame(index=expression.index),
        var=pd.DataFrame(index=expression.columns),
    )
    adata.obs["group"] = "A"
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    prior_file = (
        params["output_noisy_prior_file"] if is_noisy_prior else params["output_prior_file"]
    )
    net = pd.read_csv(data_dir / prior_file, sep="\t")
    effect_map = {"upregulates-expression": 1, "downregulates-expression": -1}
    unknown = net.loc[~net["interaction"].isin(effect_map), "interaction"].unique()
    if len(unknown):
        raise ValueError(f"Unknown interaction values found: {unknown}")
    net["interaction"] = net["interaction"].map(effect_map)
    if "weight" not in net.columns:
        net["weight"] = 1.0

    ground_truth = pd.read_csv(data_dir / params["output_ground_truth_file"], sep="\t", index_col=0)
    scores, _ = run_z_aggregate(adata, net, min_targets=MIN_TARGETS)
    adata.obsm["score_z-aggregate"] = scores.loc[adata.obs_names].reindex(
        sorted(scores.columns), axis=1
    )

    net_dc = net[["source", "target", "weight"]].copy()
    net_dc["weight"] *= net["interaction"]
    dc.mt.ulm(adata, net_dc, tmin=MIN_TARGETS)
    dc.mt.mlm(adata, net_dc, tmin=MIN_TARGETS)
    dc.mt.zscore(adata, net_dc, tmin=MIN_TARGETS)
    sc.pp.scale(adata)
    dc.mt.viper(adata, net_dc, tmin=MIN_TARGETS)

    filename_prefix = None
    if save_plot:
        plot_dir = Path(params["plot_output_dir"])
        plot_dir.mkdir(parents=True, exist_ok=True)
        prefix = "sim_roc_noisy" if is_noisy_prior else "sim_roc"
        settings = (
            f"tf-factor={params['tf_effect_factor']}"
            f"_miss={params['missing_percentage']}"
            f"_noise={params['prior_noise_percentage']}"
            f"_sigma={params['gene_propensity_sigma']}"
        )
        filename_prefix = plot_dir / f"{prefix}_{settings}"
    metrics = plot_simulated_roc(adata, ground_truth, filename_prefix=filename_prefix)
    if save_plot and not metrics.empty:
        simulation_params = {
            name: value for name, value in params.items() if not name.endswith(("_dir", "_file"))
        }
        metrics = metrics.assign(
            Experiment=plot_dir.name, Is_Noisy_Prior=is_noisy_prior, **simulation_params
        )
        metrics_file = plot_dir.parent / "roc_metrics.tsv"
        if metrics_file.exists():
            previous = pd.read_csv(metrics_file, sep="\t")
            metrics = pd.concat([previous, metrics], ignore_index=True)
            metrics = metrics.drop_duplicates(
                subset=["Experiment", "Is_Noisy_Prior", "Task", "Method", *simulation_params],
                keep="last",
            )
        metrics.to_csv(metrics_file, sep="\t", index=False)
        print(f"Saved ROC metrics to {metrics_file}")


## Shared simulation parameters

Run setup and helper cells first. Each experiment below can then run independently.


In [ ]:
BASE_PARAMS = {
    "output_dir": "simulated_data/",
    "output_exp_file": "simulated_scRNASeq.tsv",
    "output_prior_file": "simulated_prior_network.tsv",
    "output_noisy_prior_file": "simulated_noisy_prior_network.tsv",
    "output_ground_truth_file": "simulated_ground_truth.tsv",
    "n_cells": 1000,
    "n_genes": 1000,
    "n_tfs": 100,
    "target_libsize": 1e4,
    "ground_truth_active_inactive_prob": 0.1,
    "up_probability": 0.80,
    "random_seed": 42,
    "tf_effect_factor": 2,
    "missing_percentage": 75.0,
    "prior_noise_percentage": 0.0,
    "gene_propensity_sigma": 2.0,
}


## Experiment 1: Prior noise


In [ ]:
print("Running Experiment 1: Prior Noise Effect")
for noise in (0.0, 20.0, 40.0):
    params = {
        **BASE_PARAMS,
        "tf_effect_factor": 3,
        "prior_noise_percentage": noise,
        "plot_output_dir": "roc_plots/experiment1",
    }
    plot_simulated_benchmark(params, is_noisy_prior=True, save_plot=True)


## Experiment 2: Missing values


In [ ]:
print("Running Experiment 2: Missing Percentage Effect")
for missing in (60.0, 75.0, 90.0):
    params = {
        **BASE_PARAMS,
        "missing_percentage": missing,
        "plot_output_dir": "roc_plots/experiment2",
    }
    plot_simulated_benchmark(params, save_plot=True)


## Experiment 3: Gene propensity


In [ ]:
print("Running Experiment 3: Gene Propensity Sigma Effect")
for sigma in (1.0, 2.0, 3.0):
    params = {
        **BASE_PARAMS,
        "gene_propensity_sigma": sigma,
        "plot_output_dir": "roc_plots/experiment3",
    }
    plot_simulated_benchmark(params, save_plot=True)
